In [ ]:
%pip install ChemLogic seaborn

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Retrieve all runs from the experiment
runs = pd.read_csv("../experiments/runs.csv")
runs["Accuracy"] = 1 - runs["metrics.test_loss"]
mutagen_runs = runs[
    (runs["params.dataset"] == "mutagen")
    #  & (runs['params.learning_rate'] == '0.0005')
]

In [ ]:
# # Check for duplicate rows for specified columns
# columns_to_check = [col for col in runs.columns if 'param' in col or 'metric' in col]

# duplicates = runs.duplicated(subset=columns_to_check, keep='first')
# print(f"Number of duplicate rows: {duplicates.sum()}")

# # Remove duplicate rows
# runs = runs[~duplicates]
# runs = runs[
#     (runs['metrics.train_loss'].notnull()) & (runs['metrics.test_loss'].notnull())
# ]


# bare_runs = runs[runs['params.circular'].isnull()]
# full_runs = runs[runs['params.circular'].notnull()]


# models = ['gnn', 'rgcn', 'kgnn', 'kgnn_local', 'ego', 'diffusion', 'cw', 'sgn']

# for m in models:
#     print(
#         f"{m}: ({len(bare_runs[bare_runs['params.model'] == m])}, {len(full_runs[full_runs['params.model'] == m])})"
#     )

In [ ]:
print(len(mutagen_runs))
print(
    len(
        mutagen_runs[
            (mutagen_runs["metrics.test_loss"] < 0.3)
            & (mutagen_runs["metrics.train_loss"] < 0.3)
        ]
    )
)

In [ ]:
bad_runs = mutagen_runs[
    (mutagen_runs["metrics.test_loss"] > 0.3)
    & (mutagen_runs["metrics.train_loss"] > 0.3)
][["metrics.train_loss", "metrics.test_loss", "params.model", "params.architecture"]]

# Plotting train loss statistics per model and architecture
sns.boxplot(
    data=bad_runs, x="params.model", y="metrics.train_loss", hue="params.architecture"
)
plt.xlabel("Model")
plt.ylabel("Train Loss")
plt.title("Train Loss Statistics per Model and Architecture")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

# Plotting test loss statistics per model and architecture
sns.boxplot(
    data=bad_runs, x="params.model", y="metrics.test_loss", hue="params.architecture"
)
plt.xlabel("Model")
plt.ylabel("Test Loss")
plt.title("Test Loss Statistics per Model and Architecture")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

In [ ]:
mutagen_runs = pd.concat(
    [
        mutagen_runs.assign(
            loss_value=mutagen_runs["metrics.train_loss"], loss_type="Train"
        ),
        mutagen_runs.assign(
            loss_value=mutagen_runs["metrics.test_loss"], loss_type="Test"
        ),
    ]
)

# Create the grouped bar chart
g = sns.catplot(
    data=mutagen_runs,
    kind="box",
    x="loss_type",
    y="loss_value",
    hue="architecture_type",
    col="params.model",
    aspect=1,
    showfliers=False,
)
g.set_axis_labels("Models", "Loss")
g.fig.suptitle("Comparison of Train and Test Loss by Model and Architecture", y=1.05)

plt.show()

In [ ]:
mutagen_runs = mutagen_runs[
    (mutagen_runs["metrics.test_loss"] < 0.3)
    & (mutagen_runs["metrics.train_loss"] < 0.3)
]
# mutagen_runs = mutagen_runs[(mutagen_runs['metrics.test_loss'] < 0.3) & (mutagen_runs['metrics.train_loss'] < 0.3)]

# Adding a new column based on the architecture
mutagen_runs["architecture_type"] = mutagen_runs["params.architecture"].apply(
    lambda x: "bare" if x == "bare" else "enhanced"
)

# mutagen_runs = pd.concat([
#     mutagen_runs.assign(loss_value=mutagen_runs['metrics.train_loss'], loss_type='Train'),
#     mutagen_runs.assign(loss_value=mutagen_runs['metrics.test_loss'], loss_type='Test')
# ])
# mutagen_runs['model_archtype'] = mutagen_runs[['params.model', 'loss_type']].apply(lambda x: f"{x['params.model']} {x['loss_type']}", axis=1)

# mutagen_runs = mutagen_runs.sort_values(by='model_archtype')
# # Plotting train loss vs test loss for each model for bare runs as a boxplot without outliers
# plt.figure(figsize=(10, 6))
# sns.boxplot(data=mutagen_runs, x='model_archtype', y='loss_value', hue='architecture_type', showfliers=False)
# plt.xlabel('Model')
# plt.xticks(rotation='vertical')
# plt.ylabel('Loss')
# plt.title('Train Loss per Architecture Type')
# plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.show()

# Plotting train loss vs test loss for each model for bare runs as a boxplot without outliers
# plt.figure(figsize=(10, 6))
# sns.boxplot(
#     data=mutagen_runs,
#     x="params.model",
#     y="metrics.train_loss",
#     hue="architecture_type",
#     showfliers=False,
# )
# plt.xlabel("Model")
# plt.ylabel("Loss")
# plt.title("Train Loss per Architecture Type")
# plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
# plt.show()

# # Plotting test loss for each model for bare runs as a boxplot without outliers
# plt.figure(figsize=(10, 6))
# sns.boxplot(
#     data=mutagen_runs,
#     x="params.model",
#     y="metrics.test_loss",
#     hue="architecture_type",
#     showfliers=False,
# )
# plt.xlabel("Model")
# plt.ylabel("Loss")
# plt.title("Test Loss per Architecture Type")
# plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
# plt.show()


# # Plotting train loss vs test loss for each model for bare runs as a boxplot without outliers
# plt.figure(figsize=(10, 6))
# sns.boxplot(
#     data=mutagen_runs[mutagen_runs["architecture_type"] == "enhanced"],
#     x="params.model",
#     y="metrics.train_loss",
#     hue="params.architecture",
#     showfliers=False,
# )
# plt.xlabel("Model")
# plt.ylabel("Loss")
# plt.title("Train Loss per Architecture Type")
# plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
# plt.show()

# Plotting test loss for each model for bare runs as a boxplot without outliers
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=mutagen_runs[mutagen_runs["params.model"] != "aleph"],
    x="params.model",
    y="Accuracy",
    hue="params.architecture",
    showfliers=False,
)
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.title("Test Accuracy per Model and Architecture on the MUTAG Dataset")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

In [ ]:
import seaborn as sns
import pandas as pd

# Define the order and labels
hue_order = ["bare", "CCE", "CCD", "aleph"]

model_order = [
    "gnn",
    "rgcn",
    "kgnn_local",
    "kgnn",
    "ego",
    "diffusion",
    "cw",
    "sgn",
    "aleph",
]

model_labels = {
    "gnn": "GNN",
    "rgcn": "RGCN",
    "kgnn_local": "Local k-GNN",
    "kgnn": "Global k-GNN",
    "ego": "Ego GNN",
    "diffusion": "Diffusion GCN",
    "cw": "CW Network",
    "sgn": "SGN",
    "aleph": "Aleph",
}

# Filter and prepare data
plot_data = mutagen_runs.copy()

# Map model names to labels
plot_data["Model"] = plot_data["params.model"].map(model_labels)

# Create the plot
sns.boxplot(
    data=plot_data,
    x="Model",
    y="Accuracy",
    hue="params.architecture",
    hue_order=hue_order,
    order=[model_labels[m] for m in model_order],
    showfliers=False,
)

plt.ylim(0.65, 1.0)
plt.xticks(rotation=45, ha="right")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.legend(title="Mode", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(
    data=plot_data,
    x="Model",
    y="Accuracy",
    hue="params.architecture",
    hue_order=hue_order,
    order=[model_labels[m] for m in model_order],
    showfliers=False,
    ax=ax,
    palette="Set2",  # Nicer color palette
)

plt.xticks(rotation=45, ha="right")
plt.xlabel("Model Architecture", fontsize=12, fontweight="bold")
plt.ylabel("Accuracy", fontsize=12, fontweight="bold")
plt.ylim(0.65, 1.00)
plt.grid(axis="y", alpha=0.3, linestyle="--", zorder=0)  # Horizontal gridlines
plt.legend(title="Mode", bbox_to_anchor=(1.05, 1), loc="upper left", frameon=True)
plt.title(
    "Model Performance Across Architectures on MUTAG Dataset",
    fontsize=14,
    fontweight="bold",
    pad=20,
)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset order and labels
dataset_order = [
    "carcinogenous",
    "cyp2d6_substrate",
    "human_intestinal_absorption",
    "mutagen",
    "oral_bioavailability",
    "ptc",
    "ptc_fm",
    "ptc_fr",
    "ptc_mm",
    "skin_reaction",
]

dataset_labels = {
    "carcinogenous": "Carcinogenicity",
    "cyp2d6_substrate": "CYP2D6 Substrate",
    "human_intestinal_absorption": "Human Intestinal\nAbsorption",
    "mutagen": "MUTAG",
    "oral_bioavailability": "Oral\nBioavailability",
    "ptc": "PTC",
    "ptc_fm": "PTC-FM",
    "ptc_fr": "PTC-FR",
    "ptc_mm": "PTC-MM",
    "skin_reaction": "Skin Reaction",
}

hue_order = ["bare", "CCE", "CCD"]

# Filter data for these datasets and standard GNN model
plot_data = runs[
    (runs["params.dataset"].isin(dataset_order)) & (runs["params.model"] == "gnn")
].copy()

# Map dataset names to labels
plot_data["Dataset"] = plot_data["params.dataset"].map(dataset_labels)

# Create the plot
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(
    data=plot_data,
    x="Dataset",
    y="Accuracy",
    hue="params.architecture",
    hue_order=hue_order,
    order=[dataset_labels[d] for d in dataset_order],
    showfliers=False,
    ax=ax,
    palette="Set2",
)

plt.xticks(rotation=45, ha="right")
plt.xlabel("Dataset", fontsize=12, fontweight="bold")
plt.ylabel("Accuracy", fontsize=12, fontweight="bold")
plt.ylim(0.40, 1)  # Adjust based on your data range
plt.grid(axis="y", alpha=0.3, linestyle="--", zorder=0)
plt.legend(title="Mode", bbox_to_anchor=(1.05, 1), loc="upper left", frameon=True)
plt.title("Performance Across Datasets", fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Plotting train and test loss dependency on various parameters as lines
parameters = [
    "params.num_layers",
    "params.subgraph_depth",
    "params.max_depth",
    "params.parameter_size",
]

for param in parameters:
    plt.figure(figsize=(10, 6))
    sorted_mutagen_runs = mutagen_runs.sort_values(by=param)
    sns.lineplot(
        data=sorted_mutagen_runs,
        x=param,
        y="metrics.train_loss",
        label="Train Loss",
        #  style='params.architecture',
    )
    sns.lineplot(
        data=sorted_mutagen_runs,
        x=param,
        y="metrics.test_loss",
        label="Test Loss",
        #  style='params.architecture',
    )
    plt.xlabel(param)
    plt.ylabel("Loss")
    plt.title(f"Loss vs {param}")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.show()

In [ ]:
import pandas as pd

# Transforming mutagen_runs DataFrame to include loss_value and loss_type
mutagen_runs_expanded = pd.concat(
    [
        mutagen_runs.assign(
            loss_value=mutagen_runs["metrics.train_loss"], loss_type="Train"
        ),
        mutagen_runs.assign(
            loss_value=mutagen_runs["metrics.test_loss"], loss_type="Test"
        ),
    ]
)

parameters = [
    "params.oxy",
    "params.nitro",
    "params.y_shape",
    "params.hydrocarbons",
    "params.collective",
    "params.cycles",
    "params.sulfuric",
    "params.circular",
    "params.nbhoods",
    "params.paths",
    "params.relaxations",
]

for param in parameters:
    plt.figure(figsize=(10, 6))
    sns.boxplot(
        data=mutagen_runs_expanded, x="loss_type", y="loss_value", hue=param, dodge=True
    )
    plt.xlabel("loss_type")
    plt.ylabel("Loss")
    plt.title(f"Loss vs {param}")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

In [ ]:
non_mutagen_runs = runs[
    (runs["params.dataset"] != "mutagen") & (runs["params.dataset"])
    # ((runs['params.dataset'] == 'carcinogenous') | (runs['params.dataset'] == 'cyp2d6_substrate'))
]


for dataset in non_mutagen_runs["params.dataset"].unique():
    # Plotting train loss vs test loss for each model for bare runs as a boxplot
    plt.figure(figsize=(10, 6))
    sns.boxplot(
        data=non_mutagen_runs[non_mutagen_runs["params.dataset"] == dataset],
        x="params.architecture",
        y="metrics.train_loss",
        showfliers=False,
    )
    plt.xlabel("Model")
    plt.ylabel("Loss")
    plt.title(f"Train Loss per architecture on {dataset}")
    # plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.show()

    # Plotting train loss vs test loss for each model for bare runs as a boxplot
    plt.figure(figsize=(10, 6))
    sns.boxplot(
        data=non_mutagen_runs[non_mutagen_runs["params.dataset"] == dataset],
        x="params.architecture",
        y="metrics.test_loss",
        showfliers=False,
    )
    plt.xlabel("Model")
    plt.ylabel("Loss")
    plt.title(f"Test Loss per architecture on {dataset}")
    # plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.show()

In [ ]:
# show plot that shows the accuracy per dataset, that it it effective even without added rules on various datasets
non_mutagen_runs = runs[
    (runs["params.dataset"] != "anti_sarscov2_activity") & (runs["params.dataset"])
]

non_mutagen_runs = pd.concat(
    [
        non_mutagen_runs.assign(
            loss_value=non_mutagen_runs["metrics.train_loss"], loss_type="Train"
        ),
        non_mutagen_runs.assign(
            loss_value=non_mutagen_runs["metrics.test_loss"], loss_type="Test"
        ),
    ]
)


# Plotting train loss vs test loss for each model for bare runs as a boxplot
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=non_mutagen_runs,
    x="params.dataset",
    y="loss_value",
    hue="loss_type",
    showfliers=False,
)
plt.xlabel("Model")
plt.xticks(rotation="vertical")
plt.ylabel("Loss")
plt.title("Train Loss per dataset")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

In [ ]:
# Extract test loss statistics per dataset, architecture, and model
test_loss_stats = (
    runs[(runs["status"] == "FINISHED") & (runs["metrics.test_loss"] < 0.5)]
    .groupby(["params.dataset", "params.model", "params.architecture"])["Accuracy"]
    .agg(["mean", "std"])
    .reset_index()
)

test_loss_stats.columns = [
    "dataset",
    "model",
    "architecture",
    "mean_Accuracy",
    "std_Accuracy",
]
test_loss_stats["mean_Accuracy"] = test_loss_stats["mean_Accuracy"].round(4)
test_loss_stats["std_Accuracy"] = test_loss_stats["std_Accuracy"].round(4)

In [ ]:
sns.boxplot(
    data=runs[(runs["status"] == "FINISHED") & (runs["metrics.test_loss"] < 0.5)],
    x="params.model",
    y="metrics.test_loss",
    # hue='params.architecture',
)
plt.xlabel("Model")
plt.ylabel("Mean Test Loss")
plt.title("Test Loss per Model")
plt.show()

In [ ]:
test_loss_stats.to_latex("Accuracy_stats.tex", index=False)

In [ ]:
from chemlogic.utils.Pipeline import Pipeline, ArchitectureType
import pandas as pd
import numpy as np
from scipy import stats

# ===== CONFIGURATION =====
dataset_name = "mutagen"
model_name = "gnn"
architecture = "CCE"
n_runs = 10

# Fixed hyperparameters
param_size = 2
layers = 3
max_depth = 1
lr = 0.001  # Adjust as needed
epochs = 500
split = 0.7

# Rule groups to test
rule_configurations = {
    "Baseline": None,  # No rules
    "Hydrocarbons": ["hydrocarbons"],
    "Oxygen-containing": ["oxy"],
    "Nitrogen-containing": ["nitro"],
    "Sulfur-containing": ["sulfuric"],
    "Relaxations": ["relaxations"],
}

# ===== RUN ABLATION EXPERIMENTS =====
results = []

print("=" * 80)
print("RUNNING CONTROLLED ABLATION STUDY")
print("=" * 80)
print(f"Dataset: {dataset_name}")
print(f"Model: {model_name}")
print(f"Architecture: {architecture}")
print(f"Hyperparameters: layers={layers}, param_size={param_size}, lr={lr}")
print(f"Runs per configuration: {n_runs}")
print("=" * 80)
print()

for rule_name, chem_rules in rule_configurations.items():
    print(f"Testing: {rule_name}")
    print("-" * 40)

    for run_idx in range(n_runs):
        print(f"  Run {run_idx + 1}/{n_runs}...", end=" ")

        try:
            # Create pipeline
            architecture_type = ArchitectureType.from_string(architecture)
            pipeline = Pipeline(
                dataset_name=dataset_name,
                model_name=model_name,
                param_size=param_size,
                layers=layers,
                max_depth=max_depth,
                max_subgraph_depth=0,
                max_cycle_size=0,
                architecture=architecture_type,
                subgraphs=None,
                chem_rules=chem_rules,
                funnel=False,
                smiles_list=None,
                labels=None,
                task="classification",
            )

            # Run training
            train_loss, test_loss, metric, evaluator = pipeline.train_test_cycle(
                lr=lr, epochs=epochs, split_ratio=split, batches=1
            )

            # Calculate accuracy
            accuracy = 1 - test_loss

            results.append(
                {
                    "Rule Group": rule_name,
                    "Run": run_idx + 1,
                    "Test Loss": test_loss,
                    "Accuracy": accuracy,
                    "AUROC": metric,
                    "Train Loss": train_loss,
                }
            )

            print(f"Accuracy: {accuracy:.4f}, AUROC: {metric:.4f}")

        except Exception as e:
            print(f"FAILED: {e}")
            import traceback

            traceback.print_exc()
            continue

    print()

# ===== ANALYZE RESULTS =====
results_df = pd.DataFrame(results)

print("=" * 80)
print("ABLATION STUDY: Mutagen Dataset Results")
print("=" * 80)

# Calculate statistics for each rule group
summary_stats = []

baseline_accuracies = results_df[results_df["Rule Group"] == "Baseline"][
    "Accuracy"
].values
baseline_mean = baseline_accuracies.mean()
baseline_std = baseline_accuracies.std()

baseline_auroc = results_df[results_df["Rule Group"] == "Baseline"]["AUROC"].values
baseline_auroc_mean = baseline_auroc.mean()
baseline_auroc_std = baseline_auroc.std()

for rule_name in rule_configurations.keys():
    group_data = results_df[results_df["Rule Group"] == rule_name]

    if len(group_data) > 0:
        accuracies = group_data["Accuracy"].values
        auroc_scores = group_data["AUROC"].values

        mean_acc = accuracies.mean()
        std_acc = accuracies.std()
        mean_auroc = auroc_scores.mean()
        std_auroc = auroc_scores.std()
        count = len(accuracies)

        improvement_acc = mean_acc - baseline_mean
        improvement_auroc = mean_auroc - baseline_auroc_mean

        # Calculate p-values (t-test vs baseline)
        if rule_name != "Baseline":
            t_stat_acc, p_value_acc = stats.ttest_ind(baseline_accuracies, accuracies)
            t_stat_auroc, p_value_auroc = stats.ttest_ind(baseline_auroc, auroc_scores)
        else:
            p_value_acc = np.nan
            p_value_auroc = np.nan

        summary_stats.append(
            {
                "Rule Group": rule_name,
                "mean_accuracy": mean_acc,
                "std_accuracy": std_acc,
                "mean_auroc_score": mean_auroc,
                "std_auroc_score": std_auroc,
                "count": count,
                "Improvement over Baseline (Accuracy)": improvement_acc,
                "Improvement over Baseline (AUROC)": improvement_auroc,
                "P-value (Accuracy vs. Baseline)": p_value_acc,
                "P-value (AUROC vs. Baseline)": p_value_auroc,
            }
        )

summary_df = pd.DataFrame(summary_stats)
print(summary_df.to_string(index=False))
print()

# ===== SAVE RESULTS =====
results_df.to_csv("ablation_detailed_results.csv", index=False)
summary_df.to_csv("ablation_summary.csv", index=False)

print("=" * 80)
print("Results saved to:")
print("  - ablation_detailed_results.csv (all individual runs)")
print("  - ablation_summary.csv (summary statistics)")
print("=" * 80)

# ===== GENERATE LATEX TABLE =====
print()
print("=" * 80)
print("LATEX TABLE")
print("=" * 80)
print()

print("\\begin{table}[t]")
print("\\centering")
print(
    "\\caption{Controlled ablation study on the MUTAG dataset. Each configuration was trained 10 times with identical hyperparameters (Standard GNN, CCE mode, 3 layers, parameter size 2), varying only which single chemical rule group was active. Statistical significance was assessed using independent t-tests comparing each configuration against the baseline. Higher accuracy indicates better performance.}"
)
print("\\label{tab:ablation_controlled}")
print("\\begin{tabular}{lccc}")
print("\\toprule")
print("Rule Group & Accuracy & Improvement & $p$-value \\\\")
print("\\midrule")

for _, row in summary_df.iterrows():
    rule = row["Rule Group"]
    acc = row["mean_accuracy"]
    std = row["std_accuracy"]
    imp = row["Improvement over Baseline (Accuracy)"]
    p_val = row["P-value (Accuracy vs. Baseline)"]

    if rule == "Baseline":
        print(f"{rule} & ${acc:.3f} \\pm {std:.3f}$ & -- & -- \\\\")
    else:
        # Format p-value
        if p_val < 0.001:
            p_str = "$<0.001$"
        else:
            p_str = f"${p_val:.3f}$"

        # Format improvement with sign
        imp_str = f"${imp:+.3f}$" if imp != 0 else "$0.000$"

        print(f"{rule} & ${acc:.3f} \\pm {std:.3f}$ & {imp_str} & {p_str} \\\\")

print("\\bottomrule")
print("\\end{tabular}")
print("\\end{table}")

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# ===== LOAD EXISTING RESULTS =====
results_df = pd.read_csv("ablation_detailed_results.csv")

print("=" * 80)
print("ANALYZING ABLATION RESULTS")
print("=" * 80)
print()
print("NOTE: 'Test Loss' in your data is actually ACCURACY (fraction correct)")
print("      Higher accuracy = better performance")
print()

# ===== ANALYZE =====
summary_stats = []

# Get baseline statistics
baseline_data = results_df[results_df["Rule Group"] == "Baseline"]
baseline_acc = baseline_data["Test Loss"].values  # These are accuracies
baseline_mean = baseline_acc.mean()
baseline_std = baseline_acc.std()

print(f"Baseline Accuracy: {baseline_mean:.3f} ± {baseline_std:.3f}")
print()

# Analyze each rule group
for rule_name in results_df["Rule Group"].unique():
    group_data = results_df[results_df["Rule Group"] == rule_name]

    accuracies = group_data["Test Loss"].values  # Actually accuracies

    mean_acc = accuracies.mean()
    std_acc = accuracies.std()
    count = len(accuracies)

    # Improvement: current_acc - baseline_acc (positive = higher accuracy = better)
    improvement = mean_acc - baseline_mean

    # Statistical significance
    if rule_name != "Baseline":
        t_stat, p_value = stats.ttest_ind(baseline_acc, accuracies)
    else:
        p_value = np.nan

    summary_stats.append(
        {
            "Rule Group": rule_name,
            "mean_accuracy": mean_acc,
            "std_accuracy": std_acc,
            "count": count,
            "Improvement over Baseline": improvement,
            "P-value (vs. Baseline)": p_value,
        }
    )

summary_df = pd.DataFrame(summary_stats)

# Reorder to put Baseline first, then sort by improvement (descending)
baseline_row = summary_df[summary_df["Rule Group"] == "Baseline"]
other_rows = summary_df[summary_df["Rule Group"] != "Baseline"].sort_values(
    "Improvement over Baseline", ascending=False
)
summary_df = pd.concat([baseline_row, other_rows], ignore_index=True)

print("=" * 80)
print("ABLATION STUDY: Mutagen Dataset Results")
print("=" * 80)
print(summary_df.to_string(index=False))
print()

# ===== SAVE SUMMARY =====
summary_df.to_csv("ablation_summary_accuracy.csv", index=False)

# ===== GENERATE LATEX TABLE =====
print("=" * 80)
print("LATEX TABLE")
print("=" * 80)
print()

print("\\begin{table}[t]")
print("\\centering")
print(
    "\\caption{Controlled ablation study on the MUTAG dataset. Each configuration was trained 10 times with identical hyperparameters (Standard GNN, CCE mode, 3 layers, parameter size 2), varying only which single chemical rule group was active. Accuracy is the fraction of correct predictions (higher is better). Statistical significance was assessed using independent t-tests comparing each configuration against the baseline.}"
)
print("\\label{tab:ablation_controlled}")
print("\\begin{tabular}{lccc}")
print("\\toprule")
print("Rule Group & Accuracy & Improvement & $p$-value \\\\")
print("\\midrule")

for _, row in summary_df.iterrows():
    rule = row["Rule Group"]
    acc = row["mean_accuracy"]
    std = row["std_accuracy"]
    imp = row["Improvement over Baseline"]
    p_val = row["P-value (vs. Baseline)"]

    if rule == "Baseline":
        print(f"{rule} (no rules) & ${acc:.3f} \\pm {std:.3f}$ & -- & -- \\\\")
    else:
        # Format p-value
        if pd.isna(p_val):
            p_str = "--"
        elif p_val < 0.001:
            p_str = "$<0.001$"
        else:
            p_str = f"${p_val:.3f}$"

        # Format improvement (positive = higher accuracy = better)
        imp_str = f"${imp:+.3f}$"

        print(f"{rule} & ${acc:.3f} \\pm {std:.3f}$ & {imp_str} & {p_str} \\\\")

print("\\bottomrule")
print("\\end{tabular}")
print("\\end{table}")

print()
print("=" * 80)
print("INTERPRETATION")
print("=" * 80)
print()
print("Positive improvement = rule group achieved HIGHER accuracy (better)")
print("Negative improvement = rule group achieved LOWER accuracy (worse)")
print()

significant_improvements = summary_df[
    (summary_df["Rule Group"] != "Baseline")
    & (summary_df["P-value (vs. Baseline)"] < 0.05)
    & (summary_df["Improvement over Baseline"] > 0)
]

if len(significant_improvements) > 0:
    print("Significant improvements (p < 0.05):")
    for _, row in significant_improvements.iterrows():
        print(
            f"  {row['Rule Group']:25s}: {row['Improvement over Baseline']:+.4f} (p={row['P-value (vs. Baseline)']:.4f})"
        )
else:
    print("No significant improvements found.")

print()

significant_degradations = summary_df[
    (summary_df["Rule Group"] != "Baseline")
    & (summary_df["P-value (vs. Baseline)"] < 0.05)
    & (summary_df["Improvement over Baseline"] < 0)
]

if len(significant_degradations) > 0:
    print("Significant degradations (p < 0.05):")
    for _, row in significant_degradations.iterrows():
        print(
            f"  {row['Rule Group']:25s}: {row['Improvement over Baseline']:+.4f} (p={row['P-value (vs. Baseline)']:.4f})"
        )
else:
    print("No significant degradations found.")

print()
print("=" * 80)
print("BEST RULE GROUP")
print("=" * 80)
best = summary_df[summary_df["Rule Group"] != "Baseline"].iloc[0]
print(f"\nBest: {best['Rule Group']}")
print(f"Accuracy: {best['mean_accuracy']:.3f} ± {best['std_accuracy']:.3f}")
print(f"Improvement: {best['Improvement over Baseline']:+.3f}")
print(f"P-value: {best['P-value (vs. Baseline)']:.4f}")

In [ ]:
aleph_results = """
========================================
10-ROUND 70/30 ROLLING RESULTS
Individual Error Rates: [0.2962962962962963,0.1935483870967742,0.12903225806451613,0.0967741935483871,0.16666666666666666,0.24074074074074073,0.12962962962962962,0.12962962962962962,0.1111111111111111,0.16666666666666666]
Mean Error Rate (Accuracy): 0.1660
Standard Deviation: 0.0593
========================================
"""
aleph_functional_groups_results = """
========================================
10-ROUND 70/30 ROLLING RESULTS
Individual Error Rates: [0.18518518518518517,0.1774193548387097,0.1774193548387097,0.1935483870967742,0.2222222222222222,0.24074074074074073,0.16666666666666666,0.2037037037037037,0.16666666666666666,0.25925925925925924]
Mean Error Rate (Accuracy): 0.1993
Standard Deviation: 0.0303
========================================
"""

# Compare mutagen runs on All models
good_runs = mutagen_runs[
    (mutagen_runs["metrics.test_loss"] < 0.3)
    & (
        (mutagen_runs["params.model"] == "gnn")
        | (mutagen_runs["params.model"] == "aleph")
    )
]
sns.boxplot(
    data=good_runs,  # [good_runs["params.architecture"] != "bare"],
    x="params.model",
    y="metrics.test_loss",
    # hue="params.architecture",
    showfliers=False,
)
plt.xlabel("Model")
plt.ylabel("Test Loss")
plt.title("Test Loss per Model and Architecture")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

In [ ]:
new_bare = """
[Training set performance]
           Actual
        +          -
     +  91          22         113
Pred
     -  0          21          21

        91          43         134

Accuracy = 0.835820895522388
[Training set summary] [[91,22,0,21]]
[time taken] [324.1406249999982]
[total clauses constructed] [720001]
Run 10 Error Rate: 0.2037 (TP:34, FP:11, FN:0, TN:9)

========================================
10-ROUND 70/30 ROLLING RESULTS
Individual Error Rates: [0.2962962962962963,0.14516129032258066,0.14516129032258066,0.16129032258064516,0.14814814814814814,0.2222222222222222,0.12962962962962962,0.16666666666666666,0.12962962962962962,0.2037037037037037]
Mean Error Rate (Accuracy): 0.1748
Standard Deviation: 0.0496
========================================
"""
new_cce = """


"""